<a href="https://colab.research.google.com/github/rohini-th/Ai/blob/main/Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
with open(r"/content/Characters.csv","r") as file:
  data = file.read()

In [3]:
print(data[:100])

﻿Id;Name;Gender;Job;House;Wand;Patronus;Species;Blood status;Hair colour;Eye colour;Loyalty;Skills;B


**Tokenization**

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

print(tokenizer.word_index)

{'unknown': 1, 'of': 2, 'blood': 3, 'human': 4, 'male': 5, 'the': 6, 'pure': 7, 'and': 8, 'student': 9, 'female': 10, 'gryffindor': 11, 'order': 12, 'hogwarts': 13, 'blood\xa0or\xa0half': 14, 'phoenix': 15, 'school': 16, 'black': 17, 'witchcraft': 18, 'wizardry': 19, 'brown': 20, "dumbledore's": 21, 'army': 22, 'half': 23, 'dark': 24, 'slytherin': 25, 'non': 26, 'corporeal': 27, 'grey': 28, '31': 29, 'magic': 30, 'human\xa0': 31, 'ravenclaw': 32, '1': 33, '\xa01': 34, 'blue': 35, 'muggle': 36, '\xa031': 37, 'lord': 38, 'for': 39, 'hufflepuff': 40, 'red': 41, 'voldemort': 42, 'death': 43, 'september\xa01979': 44, 'in': 45, 'blonde': 46, 'to': 47, 'may': 48, 'eaters': 49, 'a': 50, '\xa0': 51, 'house': 52, 'hair': 53, 'blond': 54, '\xa01998': 55, 'skilled': 56, 'dragon': 57, 'chaser': 58, 'auror': 59, '2': 60, 'weasley': 61, 'heartstring': 62, 'magical': 63, 'august\xa01980': 64, 'head': 65, 'ghost': 66, 'green': 67, 'unicorn': 68, 'quidditch': 69, 'original': 70, 'against': 71, 'seeker':

In [5]:
print(len(tokenizer.word_index)+1)

1099


In [6]:
wordindex = tokenizer.word_index
reverse_word_index = {index:word for word,index in wordindex.items()}
print(reverse_word_index)

{1: 'unknown', 2: 'of', 3: 'blood', 4: 'human', 5: 'male', 6: 'the', 7: 'pure', 8: 'and', 9: 'student', 10: 'female', 11: 'gryffindor', 12: 'order', 13: 'hogwarts', 14: 'blood\xa0or\xa0half', 15: 'phoenix', 16: 'school', 17: 'black', 18: 'witchcraft', 19: 'wizardry', 20: 'brown', 21: "dumbledore's", 22: 'army', 23: 'half', 24: 'dark', 25: 'slytherin', 26: 'non', 27: 'corporeal', 28: 'grey', 29: '31', 30: 'magic', 31: 'human\xa0', 32: 'ravenclaw', 33: '1', 34: '\xa01', 35: 'blue', 36: 'muggle', 37: '\xa031', 38: 'lord', 39: 'for', 40: 'hufflepuff', 41: 'red', 42: 'voldemort', 43: 'death', 44: 'september\xa01979', 45: 'in', 46: 'blonde', 47: 'to', 48: 'may', 49: 'eaters', 50: 'a', 51: '\xa0', 52: 'house', 53: 'hair', 54: 'blond', 55: '\xa01998', 56: 'skilled', 57: 'dragon', 58: 'chaser', 59: 'auror', 60: '2', 61: 'weasley', 62: 'heartstring', 63: 'magical', 64: 'august\xa01980', 65: 'head', 66: 'ghost', 67: 'green', 68: 'unicorn', 69: 'quidditch', 70: 'original', 71: 'against', 72: 'seek

**Generate the Input sequences and then apply pad sequences**

In [7]:
data[:10]

'\ufeffId;Name;G'

In [8]:
token_list = tokenizer.texts_to_sequences([data])[0]
input_sequences = []

# create n gram sequences: we can create a 6 gram sequence: n=6
for i in range(5,len(token_list)):
  n_gram_sequence = token_list[i-5:i+1] #5-5=0 [0:6], 2nd loop [1:7],3rd loop: [2:8]
  input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[326, 198, 327, 328, 52, 151], [198, 327, 328, 52, 151, 329], [327, 328, 52, 151, 329, 330], [328, 52, 151, 329, 330, 3], [52, 151, 329, 330, 3, 331]]


In [9]:
max_sequences = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

'''[7, 121, 2, 1, 634, 158]
x= [7, 121, 2, 1, 634 ]
y = [158]
'''
# this scenario has become multi class classification
# Convert target to one-hot encoding
y = to_categorical(y, num_classes=len(tokenizer.word_index)+1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3758, 5)
y shape: (3758, 1099)


In [10]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequences - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(len(tokenizer.word_index)+1, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       140,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1099)           │       110,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,571 (2.02 MB)

 Trainable params: 529,571 (2.02 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=15,batch_size=32,verbose=1,callbacks=[early_stop])

Epoch 1/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.0436 - loss: 6.1633
Epoch 2/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0886 - loss: 5.4174
Epoch 3/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1288 - loss: 4.9519
Epoch 4/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1695 - loss: 4.5974
Epoch 5/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2041 - loss: 4.2838
Epoch 6/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2480 - loss: 4.0130
Epoch 7/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2610 - loss: 3.8008
Epoch 8/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2887 - loss: 3.6047
Epoch 9/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3060 - loss: 3.4286
Epoch 10/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3260 - loss: 3.2599
Epoch 11/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3358 - loss: 3.0898
Epoch 12/15
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/st

In [12]:
model.save("TextGenerationModel.keras")

In [13]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [14]:
def generate_text(seed_text, next_words=50):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequences - 1,
            padding='pre'
        ) # [0,0,0,189,45]

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = reverse_word_index.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

In [16]:
print(generate_text("Human", next_words=50))

Human beauxbatons the dark  1 september 1979  31 may 1981 86 hannah goldstein male student hufflepuff unknown non corporeal human pure blood or half blood black brown lord voldemort   death eaters skilled curse century be barty advance male slytherin unknown non corporeal human pure blood brown grey lord order of the phoenix hogwarts


In [17]:
print(generate_text("Name", next_words=50))

Name great curse a 9 lesson pre 1970s female 9 15 of durmstrang tail hair core none human pure blood or half blood black lord voldemort   death eaters skilled to century 22 11 lesson august 2 june late march blight marcus lestrange male of the department unknown non


In [18]:
print(generate_text("student", next_words=50))

student the a curse and to  1 1  31 august 1979 129 luna creevey female student gryffindor unknown


In [19]:
print(generate_text("Rohini", next_words=50))

Rohini in department academy of magical creatures and dark office professor of magic and wizarding of the to and  1 magic 31 1980  1998 dolores female weasley male slytherin unknown non corporeal human pure blood blonde brown order of the phoenix hogwarts school of witchcraft and


In [20]:
print(generate_text("pppp", next_words=50))

pppp curse to ravenclaw office slytherin unknown non corporeal human pure blood or half blood black dumbledore's army hogwarts school of witchcraft and wizardry control of the eaters duelling  31 31 11th  1996 alice weasley female student slytherin unknown
